# IFT Quantum Manifold Research Prototype (Colab Workflow)

This notebook runs a reproducible Track 2 workflow from source code.

Current status:
- Classical adaptive-manifold simulation is implemented and benchmarkable.
- A separate Classiq local arithmetic proof of concept is provided.
- The complete adaptive feedback loop has not yet been executed on quantum hardware.

This notebook does not label classical outputs as quantum results.

In [ ]:
# Clone or refresh repository
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/anthonyrenecastro-create/ift-quantum-circuit-experiment.git"
REPO_DIR = Path("/content/ift-quantum-circuit-experiment")

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", "origin/main"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print("Working directory:", REPO_DIR)
print(subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())

# Install dependencies
subprocess.run(["python", "-m", "pip", "install", "-q", "-e", ".", ".[dev]"], check=True)

## Smoke validation

In [ ]:
import subprocess

subprocess.run(["pytest", "-q"], check=True)
subprocess.run(
    [
        "python",
        "-m",
        "src.experiments.run",
        "--config",
        "configs/colab_smoke.json",
        "--mode",
        "adaptive",
        "--provider",
        "classical",
        "--output",
        "results/runs",
    ],
    check=True,
 )
subprocess.run(
    [
        "python",
        "-m",
        "src.experiments.run",
        "--config",
        "configs/colab_smoke.json",
        "--mode",
        "fixed",
        "--provider",
        "classical",
        "--output",
        "results/runs",
    ],
    check=True,
 )

## Run adaptive baseline and matched fixed control

In [ ]:
import json
import subprocess
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from src.experiments.io import reload_saved_run

subprocess.run(
    [
        "python",
        "-m",
        "src.experiments.run",
        "--config",
        "configs/adaptive_baseline.json",
        "--mode",
        "adaptive",
        "--provider",
        "classical",
        "--output",
        "results/runs",
    ],
    check=True,
 )
subprocess.run(
    [
        "python",
        "-m",
        "src.experiments.run",
        "--config",
        "configs/fixed_control.json",
        "--mode",
        "fixed",
        "--provider",
        "classical",
        "--output",
        "results/runs",
    ],
    check=True,
 )

run_dirs = sorted(Path("results/runs").glob("*"), key=lambda p: p.stat().st_mtime)
latest_two = run_dirs[-2:]
adaptive_run, fixed_run = latest_two if "adaptive" in latest_two[0].name else latest_two[::-1]

adaptive = reload_saved_run(adaptive_run)
fixed = reload_saved_run(fixed_run)

print("Adaptive run:", adaptive_run)
print("Fixed run:", fixed_run)
print("Adaptive final objective:", adaptive["summary"]["final_objective"])
print("Fixed final objective:", fixed["summary"]["final_objective"])
print("Adaptive accepted surgeries:", adaptive["summary"]["accepted_surgeries"])
print("Fixed accepted surgeries:", fixed["summary"]["accepted_surgeries"])

## Comparison plots and snapshot inspection

In [ ]:
adaptive_summary = adaptive["summary"]
fixed_summary = fixed["summary"]

labels = ["final_objective", "final_synchronization", "accepted_surgeries"]
adaptive_values = [adaptive_summary[label] for label in labels]
fixed_values = [fixed_summary[label] for label in labels]

x = np.arange(len(labels))
width = 0.35
plt.figure(figsize=(8, 4))
plt.bar(x - width / 2, adaptive_values, width, label="adaptive")
plt.bar(x + width / 2, fixed_values, width, label="fixed")
plt.xticks(x, labels)
plt.title("Adaptive vs fixed final metrics")
plt.legend()
plt.tight_layout()
plt.show()

snapshot_dir = Path(adaptive_run) / "snapshots"
snapshot_files = sorted(snapshot_dir.glob("snapshot_*.npz"))
print("Adaptive snapshot files:", [f.name for f in snapshot_files])

## Paired benchmark and output archive

In [ ]:
subprocess.run(
    [
        "python",
        "-m",
        "src.experiments.run",
        "--config",
        "configs/paired_benchmark.json",
        "--benchmark",
        "paired",
        "--provider",
        "classical",
        "--seeds",
        "0",
        "1",
        "2",
        "3",
        "4",
        "--output",
        "results/runs",
    ],
    check=True,
 )

zip_path = Path("results_runs_bundle.zip")
if zip_path.exists():
    zip_path.unlink()

subprocess.run(["zip", "-r", str(zip_path), "results/runs"], check=True)
print("Created:", zip_path.resolve())

## Optional Classiq local arithmetic section

This section is optional and separate from the classical adaptive-manifold loop.

It validates local reversible arithmetic synthesis only; it does not execute the complete adaptive manifold experiment on quantum hardware.

In [ ]:
# Optional: run only in a Classiq-enabled environment.
# !pip -q install -U classiq
# !python classiq_lorenz_step_prototype.py